# Phase 3 &mdash; Silver Layer

The Silver stage turns the faithful-but-messy Bronze table into a clean, analysis-ready dataset. Everything we do here is portable from our Phase 1 cleaning script (`src/03_data_cleaning.py`), rewritten in Spark so it can scale past single-machine memory.

### Input
`workspace.default.bronze_loans` (produced by notebook 01).

### Output
`workspace.default.silver_loans` &mdash; one row per completed loan, fewer columns, correct types, and a binary `charged_off` target.

### Operations applied (in order)
1. Filter to **completed loans only** (`Fully Paid` or `Charged Off`) and derive the `charged_off` target.
2. Drop columns with **more than 30% missing values** &mdash; too sparse to impute reliably.
3. Drop **data-leakage columns** (payment totals, recoveries, etc.) that only get populated after a loan closes and would trivially give away the outcome.
4. Drop **uninformative columns** (IDs, free-text, constant or near-constant fields).
5. **Normalize numeric strings**: strip units and percent signs from `term`, `int_rate`, `revol_util`.
6. **Parse text and dates**: map `emp_length` to an ordinal integer; convert `earliest_cr_line` and `issue_d` to dates.
7. **Null out impossible values** (e.g. negative `dti`, `revol_util > 200`) so they can be imputed in the Gold layer.

In [ ]:
try:
    spark  # type: ignore[name-defined]
except NameError:
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder.appName("phase3-silver")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
    )

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

BRONZE_TABLE = "workspace.default.bronze_loans"
SILVER_TABLE = "workspace.default.silver_loans"

In [ ]:
df = spark.table(BRONZE_TABLE)
print(f"Bronze input: {df.count():,} rows x {len(df.columns)} columns")

## Step 1 &mdash; Filter to completed loans and build the target

Only keep loans that have actually closed. Anything still `Current`, `Late`, `In Grace Period`, etc. has no final label yet, so it can't be used for supervised training.

In [ ]:
df = df.filter(F.col("loan_status").isin("Fully Paid", "Charged Off"))
df = df.withColumn("charged_off", (F.col("loan_status") == "Charged Off").cast(IntegerType()))
df = df.drop("loan_status")

total = df.count()
positives = df.filter(F.col("charged_off") == 1).count()
print(f"Completed loans: {total:,}")
print(f"  Fully Paid:   {(total - positives) / total:.1%}")
print(f"  Charged Off:  {positives / total:.1%}")

## Step 2 &mdash; Drop columns with more than 30% missing values

A column that is two-thirds blank gives the model very little to work with and is almost always an artifact of features added later in Lending Club's history. We count nulls per column in a single Spark pass and drop anything above the threshold.

In [ ]:
null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns
]).collect()[0].asDict()

high_missing = [c for c, n in null_counts.items() if n / total > 0.30]
print(f"Dropping {len(high_missing)} column(s) with more than 30% nulls.")
df = df.drop(*high_missing)
print(f"Remaining columns: {len(df.columns)}")

## Step 3 &mdash; Drop post-loan leakage columns

These fields are only populated once a loan has already closed (total payments received, recoveries, settlement information, last credit pull, etc.). Including them would give the model a trivially perfect signal, so they have to go.

In [ ]:
leakage_cols = [
    "last_pymnt_amnt", "last_pymnt_d",
    "total_pymnt", "total_pymnt_inv",
    "total_rec_prncp", "total_rec_int", "total_rec_late_fee",
    "out_prncp", "out_prncp_inv",
    "hardship_flag",
    "recoveries", "collection_recovery_fee",
    "last_credit_pull_d",
    "last_fico_range_low", "last_fico_range_high",
    "debt_settlement_flag", "debt_settlement_flag_date",
    "settlement_status", "settlement_date",
    "settlement_amount", "settlement_percentage", "settlement_term",
]
existing = [c for c in leakage_cols if c in df.columns]
df = df.drop(*existing)
print(f"Dropped {len(existing)} leakage column(s). Remaining: {len(df.columns)}")

## Step 4 &mdash; Drop uninformative columns

- IDs (`id`, `member_id`, `url`) &mdash; can't generalize across rows.
- Free text (`desc`, `emp_title`, `title`) &mdash; too noisy for this project's scope.
- Near-duplicates of kept columns (`funded_amnt` vs `loan_amnt`).
- Constant or near-constant columns in 2014&ndash;2017 data (`policy_code`, `pymnt_plan`).
- `grade` &mdash; almost perfectly correlated with `int_rate`; we keep `int_rate`.

In [ ]:
uninformative_cols = [
    "id", "member_id", "url",
    "desc", "title", "emp_title",
    "zip_code",
    "policy_code", "pymnt_plan",
    "funded_amnt", "funded_amnt_inv",
    "next_pymnt_d",
    "installment",
    "disbursement_method", "application_type",
    "Unnamed: 0", "unnamed_0",
    "grade",
    "tax_liens", "acc_now_delinq", "num_sats", "delinq_amnt",
    "num_tl_30dpd", "chargeoff_within_12_mths",
    "collections_12_mths_ex_med", "num_tl_120dpd_2m",
]
existing = [c for c in uninformative_cols if c in df.columns]
df = df.drop(*existing)
print(f"Dropped {len(existing)} uninformative column(s). Remaining: {len(df.columns)}")

## Step 5 &mdash; Normalize numeric strings

A few columns are stored as strings with embedded units. Strip the units and cast to proper numeric types.

| Column | Before | After |
|---|---|---|
| `term`       | `" 36 months"` | `36` (int) |
| `int_rate`   | `"12.5%"`      | `12.5` (float) |
| `revol_util` | `"45.3%"`      | `45.3` (float) |

In [ ]:
if "term" in df.columns:
    df = df.withColumn(
        "term",
        F.regexp_extract(F.trim(F.col("term").cast("string")), r"(\d+)", 1).cast(IntegerType()),
    )

for col in ["int_rate", "revol_util"]:
    if col in df.columns:
        df = df.withColumn(
            col,
            F.regexp_replace(F.col(col).cast("string"), "%", "").cast(DoubleType()),
        )

df.select("term", "int_rate", "revol_util").show(3)

## Step 6 &mdash; Parse text and date columns

- `emp_length`: `"< 1 year"` -> 0, `"5 years"` -> 5, `"10+ years"` -> 10, missing -> -1. The -1 sentinel keeps the "missing" signal without introducing NaNs that downstream stages would need to impute.
- `earliest_cr_line`: `"Oct-1981"` -> a real `date`.
- `issue_d`: convert from string to `date` so we can filter / sort chronologically in the Gold layer.

In [ ]:
if "emp_length" in df.columns:
    df = df.withColumn(
        "emp_length",
        F.when(F.col("emp_length") == "< 1 year", 0)
         .when(F.col("emp_length") == "10+ years", 10)
         .when(F.col("emp_length").isNull(), -1)
         .otherwise(F.regexp_extract(F.col("emp_length"), r"(\d+)", 1).cast(IntegerType()))
    )
    df = df.fillna({"emp_length": -1})

if "earliest_cr_line" in df.columns:
    df = df.withColumn(
        "earliest_cr_line",
        F.to_date(F.col("earliest_cr_line").cast("string"), "MMM-yyyy"),
    )

if "issue_d" in df.columns:
    df = df.withColumn("issue_d", F.to_date(F.col("issue_d").cast("string")))

df.select("emp_length", "earliest_cr_line", "issue_d").show(3)

## Step 7 &mdash; Null out impossible values

A handful of rows have values that are physically impossible (negative debt-to-income, negative annual income, revolving utilization above 200%). We set those to `NULL` so the Gold layer's median imputation handles them, rather than letting impossible values contaminate the model.

In [ ]:
impossible_rules = [
    ("dti",        F.col("dti") < 0),
    ("annual_inc", F.col("annual_inc") < 0),
    ("open_acc",   F.col("open_acc") < 0),
    ("revol_util", F.col("revol_util") > 200),
]
for col, cond in impossible_rules:
    if col in df.columns:
        df = df.withColumn(col, F.when(cond, None).otherwise(F.col(col)))
print("Applied domain constraints.")

## Step 8 &mdash; Write the Silver Delta table

In [ ]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

final = spark.table(SILVER_TABLE)
print(f"Wrote Delta table: {SILVER_TABLE}")
print(f"Final shape: {final.count():,} rows x {len(final.columns)} columns")

In [ ]:
spark.table(SILVER_TABLE).limit(5).toPandas()